# 13.4 - Latency Optimization

**Module 13 - Cost & Performance** | Netsetos GenAI Engineering

This notebook builds latency intuition with one tiny, keyless model: a `ttft`/`total` pair of functions that split every request into a first-token wait (prefill) and a per-output-token grind (decode). From that one model it derives the whole latency toolkit - why streaming wins, which fix hits which phase, how parallelism and caching remove waiting, and how a hedged request tames the tail.

Read top to bottom: each code cell has a short **intro above** it and a **step-by-step walkthrough below** it. Run the cell, then check its output against the walkthrough.

## Setup

This notebook is pure Python arithmetic - no API calls, no model, no key. Every cell is a small latency model you can run offline. This first cell just notes the optional install and that the illustrative timings are fixed, not random.

In [ ]:
# Install dependencies (if running in Colab or fresh environment)
# Uncomment the next line if needed:
# !pip install anthropic -q

# Reproducibility - the lesson uses seeded random values throughout

**Explanation**

A no-op prep cell. There is nothing to import for the exercises - the whole notebook runs on plain functions and `print`. The commented `pip install anthropic` is only for the one illustrative `latency-stack.py` snippet shown later in markdown, which is never executed here.

**How the code works - step by step**
- **Commented `pip install anthropic`** - uncomment only if you want to run the illustrative streaming stack shown later (it is not run in this notebook).
- **Reproducibility comment** - every timing in the notebook is a fixed, hand-chosen constant (milliseconds), so runs are identical - nothing here is actually random.

**In one line:** environment prep - no imports needed, the exercises are self-contained arithmetic.

**What you'll see:** (no output - environment prep)

## 1 - Latency has two numbers, and the user feels TTFT

Every LLM request splits into two times: **time-to-first-token (TTFT)**, the wait before anything appears, and **decode**, the output tokens times the per-token cost. The reframe that drives the whole lesson is that a user on a screen feels only TTFT - the moment output starts - not the total. This cell prices both, then shows the tail (p50 vs p99) that a fast median hides.

In [ ]:
# Illustrative latency model (ms). PREFILL sets TTFT (grows with INPUT); DECODE sets the rest (per OUTPUT token).
def ttft(in_tok, per_k=200, base=150):     # time-to-first-token = prefill (compute-bound)
    return base + in_tok / 1000 * per_k
TPOT = 20                                  # time-per-output-token = decode (memory-bound), ms/token
def total(in_tok, out_tok, per_k=200, tpot=TPOT):
    return ttft(in_tok, per_k) + out_tok * tpot
# Latency has two numbers, and the user only FEELS one of them: TTFT (when output starts).
in_tok, out_tok = 1000, 200
t_ttft = ttft(in_tok); t_decode = out_tok * TPOT; t_total = total(in_tok, out_tok)
print("One request (in {} / out {} tokens):".format(in_tok, out_tok))
print("  TTFT (first token):  {:.0f} ms   <- what the user FEELS".format(t_ttft))
print("  decode (rest):       {:.0f} ms   ({} tokens x {} ms)".format(t_decode, out_tok, TPOT))
print("  total:               {:.0f} ms".format(t_total))
print()
print("With streaming the user starts reading at {:.0f} ms; without it they stare at a spinner for {:.0f} ms.".format(t_ttft, t_total))
print("Same total work, but the felt wait is {:.1f}x shorter when you stream.".format(t_total / t_ttft))
print()
# The tail is the other half of the story: LLM p50/p99 runs 1:8 to 1:15, not 1:2 like a REST API.
p50_ttft, p99_ttft = 350, 2800
print("And measure BOTH ends: p50 TTFT {} ms vs p99 TTFT {} ms = {:.0f}x.".format(p50_ttft, p99_ttft, p99_ttft / p50_ttft))
print("A fast median hides a slow tail. Log TTFT and total separately, at p50 AND p99.")

# Output:
# One request (in 1000 / out 200 tokens):
#   TTFT (first token):  350 ms   <- what the user FEELS
#   decode (rest):       4000 ms   (200 tokens x 20 ms)
#   total:               4350 ms
#
# With streaming the user starts reading at 350 ms; without it they stare at a spinner for 4350 ms.
# Same total work, but the felt wait is 12.4x shorter when you stream.
#
# And measure BOTH ends: p50 TTFT 350 ms vs p99 TTFT 2800 ms = 8x.
# A fast median hides a slow tail. Log TTFT and total separately, at p50 AND p99.

**Explanation**

This is the notebook's foundation: two functions, `ttft` and `total`, that model a request as prefill (a fixed base plus a per-1000-input-token cost) followed by decode (output tokens times a flat 20 ms each). It is a measurement lens, not a model call - it computes what the user *feels* (TTFT) versus what actually runs (total), then contrasts a p50 and p99 TTFT to make the point that one number is never enough.

**How the code works - step by step**
- **`ttft(in_tok)`** - prefill time = `base` (150 ms) plus `in_tok / 1000 * per_k` (200 ms per 1000 input tokens); this is the first-token wait and it grows with the *input*.
- **`TPOT = 20`** - time-per-output-token, the flat decode cost of each generated token.
- **`total(in_tok, out_tok)`** - `ttft(...) + out_tok * TPOT`; the full request time.
- **Decompose one request** - for 1000 in / 200 out it computes TTFT (350 ms), decode (4000 ms), and total (4350 ms), labelling TTFT as "what the user FEELS".
- **The streaming preview** - divides total by TTFT to show the felt wait is ~12x shorter when output starts early.
- **The tail** - hard-codes a p50 TTFT of 350 ms and a p99 of 2800 ms and prints the 8x ratio, the reason to log both.

**In one line:** split a request into felt (TTFT) and total, then show the tail is a separate number you must also watch.

**What you'll see:** TTFT 350 ms, decode 4000 ms, total 4350 ms; a note that streaming makes the felt wait 12.4x shorter for the same work; and p50 350 ms vs p99 2800 ms = 8x, with the rule to log TTFT and total separately at p50 and p99.

## 2 - Stream the output

Streaming is the highest-ROI latency move because it changes the *felt* wait, not the work. A blocking call withholds the whole answer until it is done; a streaming call hands each token to the UI as it is generated, so the user starts reading at the first token. This cell prices the same request delivered both ways.

In [ ]:
# Illustrative latency model (ms). PREFILL sets TTFT (grows with INPUT); DECODE sets the rest (per OUTPUT token).
def ttft(in_tok, per_k=200, base=150):     # time-to-first-token = prefill (compute-bound)
    return base + in_tok / 1000 * per_k
TPOT = 20                                  # time-per-output-token = decode (memory-bound), ms/token
def total(in_tok, out_tok, per_k=200, tpot=TPOT):
    return ttft(in_tok, per_k) + out_tok * tpot
# Streaming is the single highest-ROI latency move - it changes the FELT wait, not the total work.
in_tok, out_tok = 1000, 200
t_ttft = ttft(in_tok); t_total = total(in_tok, out_tok)
# Blocking: the whole answer is withheld until it is finished. Streaming: tokens appear as generated.
felt_blocking = t_total     # user sees nothing until the end
felt_streaming = t_ttft     # user starts reading at the first token, then reads as it streams
print("Same request, two delivery modes (total compute is identical at {:.0f} ms):".format(t_total))
print("  blocking  -> first thing on screen at {:.0f} ms (the full answer, all at once)".format(felt_blocking))
print("  streaming -> first thing on screen at {:.0f} ms (then it fills in as you read)".format(felt_streaming))
print()
print("Perceived wait drops {:.1f}x ({:.0f} ms -> {:.0f} ms) for ZERO change in total time or cost.".format(
    felt_blocking / felt_streaming, felt_blocking, felt_streaming))
print("Once tokens arrive faster than a person reads (~{} ms/token here), the user never waits again after the first.".format(TPOT))
print("Stream first. It is the cheapest, biggest win in latency - nothing else touches perceived speed like it.")

# Output:
# Same request, two delivery modes (total compute is identical at 4350 ms):
#   blocking  -> first thing on screen at 4350 ms (the full answer, all at once)
#   streaming -> first thing on screen at 350 ms (then it fills in as you read)
#
# Perceived wait drops 12.4x (4350 ms -> 350 ms) for ZERO change in total time or cost.
# Once tokens arrive faster than a person reads (~20 ms/token here), the user never waits again after the first.
# Stream first. It is the cheapest, biggest win in latency - nothing else touches perceived speed like it.

**Explanation**

A before/after comparison built on the exact same `ttft`/`total` model from cell 1. It makes explicit that blocking and streaming have identical total compute - the only thing that changes is *when the first thing lands on screen*. This is the cheapest, largest perceived-latency win, and the cell exists to quantify it.

**How the code works - step by step**
- **Redefines `ttft`/`TPOT`/`total`** - each cell is standalone, so the model is repeated verbatim.
- **Compute TTFT and total** - for the 1000 in / 200 out request (350 ms and 4350 ms).
- **`felt_blocking = t_total`** - blocking shows nothing until the end, so the felt wait is the full 4350 ms.
- **`felt_streaming = t_ttft`** - streaming shows the first token at 350 ms, so that is the felt wait.
- **The drop** - divides blocking by streaming to print the 12.4x perceived improvement for zero change in total time or cost.

**In one line:** same total compute, but streaming makes the felt wait TTFT instead of total - a ~12x win for free.

**What you'll see:** Both modes total 4350 ms of compute, but the first thing on screen is 4350 ms (blocking) vs 350 ms (streaming) - a 12.4x drop in perceived wait, with the reminder that once tokens outpace reading the user never waits again.

## 3 - Prefill vs decode: where the time goes

To cut the *real* time you have to know where it goes, and inference has two phases. **Prefill** reads the whole prompt at once (compute-bound, builds the KV cache, sets TTFT, grows with input); **decode** writes one token at a time (memory-bound, reuses the cache, sets the rest, grows with output). This cell profiles two requests to show which phase dominates.

In [ ]:
# Illustrative latency model (ms). PREFILL sets TTFT (grows with INPUT); DECODE sets the rest (per OUTPUT token).
def ttft(in_tok, per_k=200, base=150):     # time-to-first-token = prefill (compute-bound)
    return base + in_tok / 1000 * per_k
TPOT = 20                                  # time-per-output-token = decode (memory-bound), ms/token
def total(in_tok, out_tok, per_k=200, tpot=TPOT):
    return ttft(in_tok, per_k) + out_tok * tpot
# WHERE the time goes decides WHICH fix helps. Prefill (input) sets TTFT; decode (output) sets the rest.
for label, in_tok, out_tok in [("short chat", 500, 150), ("RAG (big context)", 4000, 150)]:
    p = ttft(in_tok); d = out_tok * TPOT; t = p + d
    print("{:<18} in {:>4} / out {:>3}:  prefill/TTFT {:>4.0f} ms  +  decode {:>4.0f} ms  =  {:>4.0f} ms total".format(
        label, in_tok, out_tok, p, d, t))
print()
print("Prefill processes the WHOLE prompt at once (compute-bound) and builds the KV cache - it grows with INPUT.")
print("Decode generates one token at a time (memory-bound), reusing the KV cache - it grows with OUTPUT.")
print("So: a big RAG context blows up TTFT (prefill), while a long answer blows up total (decode).")
print("Match the fix to the phase - caching and a shorter prompt cut prefill; shorter output and speculation cut decode.")

# Output:
# short chat         in  500 / out 150:  prefill/TTFT  250 ms  +  decode 3000 ms  =  3250 ms total
# RAG (big context)  in 4000 / out 150:  prefill/TTFT  950 ms  +  decode 3000 ms  =  3950 ms total
#
# Prefill processes the WHOLE prompt at once (compute-bound) and builds the KV cache - it grows with INPUT.
# Decode generates one token at a time (memory-bound), reusing the KV cache - it grows with OUTPUT.
# So: a big RAG context blows up TTFT (prefill), while a long answer blows up total (decode).
# Match the fix to the phase - caching and a shorter prompt cut prefill; shorter output and speculation cut decode.

**Explanation**

A diagnostic, not a fix. It loops over two contrasting workloads and breaks each into its prefill and decode components using the same model, so you can see that a big context inflates TTFT while a long answer inflates the total. This is the decision rule the next two cells depend on - match the fix to the phase.

**How the code works - step by step**
- **Redefines the model** - `ttft`/`TPOT`/`total` again.
- **Loops two workloads** - `("short chat", 500, 150)` and `("RAG (big context)", 4000, 150)`.
- **Per row** - computes `p = ttft(in_tok)` (prefill/TTFT), `d = out_tok * TPOT` (decode), and `t = p + d` (total), printed in an aligned table.
- **The takeaway prints** - prefill grows with input and builds the KV cache; decode grows with output and reuses it; a big RAG context blows up TTFT while a long answer blows up total.

**In one line:** profile a request into prefill (input, TTFT) and decode (output, rest) so you know which lever to pull.

**What you'll see:** A two-row table: short chat = prefill 250 ms + decode 3000 ms = 3250 ms; RAG big context = prefill 950 ms + decode 3000 ms = 3950 ms - showing the RAG context inflates only the prefill/TTFT side, with the rule to match each fix to its phase.

## 4 - Cut TTFT: prompt caching, a shorter prompt, a faster model

If TTFT is your bottleneck you are looking at prefill, and there are three levers. **Prompt caching** reuses the cached KV state for a stable prefix so only the fresh tokens are prefilled - usually the biggest RAG win. A **shorter prompt** is less to prefill, and a **faster model** prefills quicker. All three help prefill only. This cell prices them on a RAG prompt.

In [ ]:
# Illustrative latency model (ms). PREFILL sets TTFT (grows with INPUT); DECODE sets the rest (per OUTPUT token).
def ttft(in_tok, per_k=200, base=150):     # time-to-first-token = prefill (compute-bound)
    return base + in_tok / 1000 * per_k
TPOT = 20                                  # time-per-output-token = decode (memory-bound), ms/token
def total(in_tok, out_tok, per_k=200, tpot=TPOT):
    return ttft(in_tok, per_k) + out_tok * tpot
# Cut TTFT = cut PREFILL. Three levers: prompt caching, a shorter prompt, and a faster model.
stable_prefix, fresh = 3500, 500          # a RAG prompt: a stable system+context prefix + a fresh query
cold = ttft(stable_prefix + fresh)        # cold: prefill the whole 4000-token prompt
cached = ttft(fresh) + 30                 # cached: the 3500 prefix is reused, only the 500 fresh tokens prefill
faster_model = ttft(stable_prefix + fresh, per_k=100)   # a faster/smaller model prefills quicker
print("TTFT for a {}-token RAG prompt (stable prefix {} + fresh {}):".format(stable_prefix + fresh, stable_prefix, fresh))
print("  cold (no cache):        {:.0f} ms".format(cold))
print("  prompt caching:         {:.0f} ms   ({:.1f}x faster - the stable prefix is not re-processed)".format(cached, cold / cached))
print("  faster model (prefill): {:.0f} ms   ({:.1f}x faster)".format(faster_model, cold / faster_model))
print()
print("Prompt caching is usually the biggest single TTFT win for RAG and long context (it skips the expensive prefill).")
print("Caching helps ONLY prefill/TTFT - it does nothing for the decode phase. Pick the lever that hits your phase.")

# Output:
# TTFT for a 4000-token RAG prompt (stable prefix 3500 + fresh 500):
#   cold (no cache):        950 ms
#   prompt caching:         280 ms   (3.4x faster - the stable prefix is not re-processed)
#   faster model (prefill): 550 ms   (1.7x faster)
#
# Prompt caching is usually the biggest single TTFT win for RAG and long context (it skips the expensive prefill).
# Caching helps ONLY prefill/TTFT - it does nothing for the decode phase. Pick the lever that hits your phase.

**Explanation**

A what-if calculator over the prefill side of the model. It takes a RAG prompt (a stable 3500-token prefix plus a 500-token fresh query) and recomputes TTFT under three interventions, so you can see caching's outsized effect. The key teaching point baked into the numbers: every lever here moves TTFT and none of them touches decode.

**How the code works - step by step**
- **`stable_prefix, fresh = 3500, 500`** - a RAG prompt split into a repeating prefix and a fresh query.
- **`cold`** - `ttft(stable_prefix + fresh)`: prefill the whole 4000-token prompt from scratch.
- **`cached`** - `ttft(fresh) + 30`: the stable prefix is reused, so only the 500 fresh tokens prefill (plus a small cache-lookup overhead).
- **`faster_model`** - `ttft(..., per_k=100)`: halves the per-token prefill cost to model a smaller/faster model.
- **Prints the three TTFTs and their speedups** - and stresses that caching helps prefill/TTFT only, nothing for decode.

**In one line:** on a RAG prompt, caching skips the stable prefix's prefill for the biggest TTFT drop; a faster model and shorter prompt help too - all prefill-only.

**What you'll see:** For the 4000-token RAG prompt: cold 950 ms, prompt caching 280 ms (3.4x faster), faster model 550 ms (1.7x faster) - with the reminder that caching is the biggest single RAG TTFT win and helps prefill only.

## 5 - Cut decode: shorter output and speculative decoding

If a long answer is your bottleneck you are looking at decode, and total = TTFT + output x per-token time - so cut the output or speed up each token. A **shorter answer** cuts total directly. **Speculative decoding** has a small draft model propose tokens the target verifies in one pass - ~2-3x faster decode with identical output - but it cannot cut TTFT. This cell prices both.

In [ ]:
# Illustrative latency model (ms). PREFILL sets TTFT (grows with INPUT); DECODE sets the rest (per OUTPUT token).
def ttft(in_tok, per_k=200, base=150):     # time-to-first-token = prefill (compute-bound)
    return base + in_tok / 1000 * per_k
TPOT = 20                                  # time-per-output-token = decode (memory-bound), ms/token
def total(in_tok, out_tok, per_k=200, tpot=TPOT):
    return ttft(in_tok, per_k) + out_tok * tpot
# Cut total = cut DECODE. total = TTFT + out x TPOT, so cut the output tokens, or speed up decode.
in_tok = 1000; t_ttft = ttft(in_tok)
verbose, short = 200, 80
t_verbose = t_ttft + verbose * TPOT
t_short = t_ttft + short * TPOT
# Speculative decoding: a small draft model proposes tokens, the target verifies them in one pass.
# ~2.5 tokens accepted per step -> effective decode ~2.5x faster. It speeds DECODE, not prefill/TTFT.
tpot_spec = TPOT / 2.5
t_spec = t_ttft + verbose * tpot_spec
print("total = TTFT ({:.0f} ms) + output x {} ms/token:".format(t_ttft, TPOT))
print("  verbose answer ({} tok):        {:.0f} ms".format(verbose, t_verbose))
print("  structured/short answer ({} tok): {:.0f} ms   ({:.1f}x faster - fewer tokens to decode)".format(short, t_short, t_verbose / t_short))
print("  speculative decoding ({} tok):   {:.0f} ms   ({:.1f}x faster decode; TTFT unchanged at {:.0f} ms)".format(
    verbose, t_spec, t_verbose / t_spec, t_ttft))
print()
print("Shorter output cuts total directly (and cost - Lesson 13.2). Speculation cuts the per-token decode time.")
print("Speculative decoding helps DECODE only - it cannot cut TTFT, because prefill still has to run first.")

# Output:
# total = TTFT (350 ms) + output x 20 ms/token:
#   verbose answer (200 tok):        4350 ms
#   structured/short answer (80 tok): 1950 ms   (2.2x faster - fewer tokens to decode)
#   speculative decoding (200 tok):   1950 ms   (2.2x faster decode; TTFT unchanged at 350 ms)
#
# Shorter output cuts total directly (and cost - Lesson 13.2). Speculation cuts the per-token decode time.
# Speculative decoding helps DECODE only - it cannot cut TTFT, because prefill still has to run first.

**Explanation**

The decode-side twin of cell 4: a what-if calculator that holds TTFT fixed and varies the decode component. It contrasts a verbose answer against a shorter one and against speculative decoding (modelled by dividing TPOT by 2.5), making the crucial point that speculation shrinks the decode bar but leaves TTFT untouched because prefill still has to run first.

**How the code works - step by step**
- **`in_tok = 1000; t_ttft = ttft(in_tok)`** - fixes TTFT at 350 ms for all three variants.
- **`t_verbose`** - `t_ttft + verbose * TPOT` for a 200-token answer.
- **`t_short`** - same but for an 80-token answer - fewer tokens to decode cuts total directly.
- **`tpot_spec = TPOT / 2.5`** - speculative decoding accepts ~2.5 proposed tokens per verify step, so effective per-token decode is 2.5x faster.
- **`t_spec`** - `t_ttft + verbose * tpot_spec`: the full 200-token answer but at the faster decode rate; TTFT stays at 350 ms.
- **Prints all three** - and notes speculation helps decode only, and that a shorter output also saves cost (Lesson 13.2).

**In one line:** cut total by generating fewer tokens or by speeding each token (speculative decoding) - both leave TTFT untouched.

**What you'll see:** TTFT 350 ms fixed; verbose 200-token answer 4350 ms, short 80-token answer 1950 ms (2.2x faster), speculative decoding on the full 200 tokens 1950 ms (2.2x faster decode) - with TTFT unchanged, underscoring speculation is decode-only.

## 6 - Do less waiting: parallel calls and semantic caching

The fastest work is the work you avoid. Two techniques remove waiting outright: run **independent** calls concurrently so wall-clock is the *max* not the *sum* (N calls in the time of the slowest), and serve repeated or near-identical questions from a **semantic cache** in milliseconds, matched by intent rather than exact string. This cell prices both.

In [ ]:
# Illustrative latency model (ms). PREFILL sets TTFT (grows with INPUT); DECODE sets the rest (per OUTPUT token).
def ttft(in_tok, per_k=200, base=150):     # time-to-first-token = prefill (compute-bound)
    return base + in_tok / 1000 * per_k
TPOT = 20                                  # time-per-output-token = decode (memory-bound), ms/token
def total(in_tok, out_tok, per_k=200, tpot=TPOT):
    return ttft(in_tok, per_k) + out_tok * tpot
# Do less waiting: run INDEPENDENT calls in parallel (wall-clock = max, not sum), and cache repeats.
per_call = 1200; n = 4
serial = n * per_call        # four calls one after another
parallel = per_call          # four independent calls at once -> as slow as the slowest one
print("Four INDEPENDENT model calls at {} ms each:".format(per_call))
print("  serial (one after another): {} ms".format(serial))
print("  parallel (all at once):     {} ms   ({}x faster - wall-clock is the MAX, not the sum)".format(parallel, serial // parallel))
print()
# Semantic caching: a repeated or near-identical question returns a stored answer in milliseconds.
hit_rate, cache_ms = 0.73, 5      # illustrative high-repetition workload
mean = hit_rate * cache_ms + (1 - hit_rate) * per_call
print("Semantic cache on a repetitive workload (hit rate {:.0f}%, a hit returns in {} ms):".format(hit_rate * 100, cache_ms))
print("  mean latency: {} ms -> {:.0f} ms   ({:.1f}x faster; a cache hit skips the model entirely)".format(per_call, mean, per_call / mean))
print("Parallelize what is independent; cache what repeats. The fastest call is the one you never make.")

# Output:
# Four INDEPENDENT model calls at 1200 ms each:
#   serial (one after another): 4800 ms
#   parallel (all at once):     1200 ms   (4x faster - wall-clock is the MAX, not the sum)
#
# Semantic cache on a repetitive workload (hit rate 73%, a hit returns in 5 ms):
#   mean latency: 1200 ms -> 328 ms   (3.7x faster; a cache hit skips the model entirely)
# Parallelize what is independent; cache what repeats. The fastest call is the one you never make.

**Explanation**

A step away from the single-request model into whole-request orchestration. The first half is a serial-vs-parallel comparison of four independent calls; the second half is an expected-value calculation for a semantic cache, blending a fast cache-hit time with the full model time by hit rate. It quantifies why avoiding or overlapping work beats optimizing any single call.

**How the code works - step by step**
- **`per_call = 1200; n = 4`** - four independent model calls at 1200 ms each.
- **`serial = n * per_call`** - run one after another, you pay the sum (4800 ms).
- **`parallel = per_call`** - run concurrently, wall-clock is the slowest single call (1200 ms).
- **Prints the 4x speedup** - wall-clock is the max, not the sum (only independent calls qualify).
- **`hit_rate, cache_ms = 0.73, 5`** - a repetitive workload: 73% of queries hit the cache, a hit returns in 5 ms.
- **`mean = hit_rate * cache_ms + (1 - hit_rate) * per_call`** - expected latency blending fast hits with full-cost misses.
- **Prints the drop** - mean falls from 1200 ms to 328 ms because a hit skips the model entirely.

**In one line:** parallelize what is independent (wall-clock = max), cache what repeats (a hit skips the model) - the fastest call is the one you never make.

**What you'll see:** Four independent calls: 4800 ms serial vs 1200 ms parallel (4x faster). Semantic cache at 73% hit rate: mean latency 1200 ms -> 328 ms (3.7x faster), because most requests never reach the model.

## 7 - Defend the tail

Everything so far optimized the median. But a predictable slice of users lands on the **tail**, and on LLMs p99 runs 8-15x the median. You set a latency budget (an SLO), measure p99, and defend it with a hard timeout, a **hedged request** (fire a second attempt after p50, take whichever answers first), and a fast fallback. This cell prices the tail defense.

In [ ]:
# Illustrative latency model (ms). PREFILL sets TTFT (grows with INPUT); DECODE sets the rest (per OUTPUT token).
def ttft(in_tok, per_k=200, base=150):     # time-to-first-token = prefill (compute-bound)
    return base + in_tok / 1000 * per_k
TPOT = 20                                  # time-per-output-token = decode (memory-bound), ms/token
def total(in_tok, out_tok, per_k=200, tpot=TPOT):
    return ttft(in_tok, per_k) + out_tok * tpot
# The tail is what a slice of users ALWAYS hit. Set a budget, measure p99, and defend it.
p50, p99 = 400, 4000              # TTFT ms; LLM p50/p99 runs 1:8 to 1:15 (not 1:2 like a REST API)
BUDGET_P99 = 800                  # an SLO: 800 ms TTFT at p99
print("TTFT distribution: p50 {} ms, p99 {} ms  = {:.0f}x (LLM tails are long).".format(p50, p99, p99 / p50))
print("SLO: {} ms TTFT at p99 -> the raw p99 of {} ms BREACHES it by {:.1f}x.".format(BUDGET_P99, p99, p99 / BUDGET_P99))
print()
# Defend the tail: a hard timeout, a HEDGED request (fire a second attempt after p50, take the first to answer),
# and a fast fallback model. Hedging bounds the tail near ~2x p50 instead of the raw p99.
p99_hedged = 2 * p50 + 100        # fire the hedge at p50, the faster of the two usually wins
print("Defend it: timeout at the budget + a HEDGED request (fire a 2nd attempt at p50, take the first) + a fast fallback.")
print("  p99 with hedging: {} ms -> {} ms   (back near the {} ms budget)".format(p99, p99_hedged, BUDGET_P99))
print("Optimize the median with streaming and caching; defend the tail with timeouts, hedging, and a fallback (Lesson 12.2).")

# Output:
# TTFT distribution: p50 400 ms, p99 4000 ms  = 10x (LLM tails are long).
# SLO: 800 ms TTFT at p99 -> the raw p99 of 4000 ms BREACHES it by 5.0x.
#
# Defend it: timeout at the budget + a HEDGED request (fire a 2nd attempt at p50, take the first) + a fast fallback.
#   p99 with hedging: 4000 ms -> 900 ms   (back near the 800 ms budget)
# Optimize the median with streaming and caching; defend the tail with timeouts, hedging, and a fallback (Lesson 12.2).

**Explanation**

The final piece: a tail-latency calculator that shows a raw p99 breaching an SLO and then how a hedge pulls it back. It models the hedge as `2 * p50 + 100` - firing a second attempt at the median and taking the faster of the two, which bounds the tail near twice the p50 instead of the raw p99. This is the reliability toolkit from Lesson 12.2 applied to latency.

**How the code works - step by step**
- **`p50, p99 = 400, 4000`** - a TTFT distribution with a 10x tail, typical of LLM APIs.
- **`BUDGET_P99 = 800`** - the SLO: 800 ms TTFT at p99.
- **Prints the breach** - the raw p99 of 4000 ms exceeds the 800 ms budget by 5x.
- **`p99_hedged = 2 * p50 + 100`** - fire a hedge at p50; the faster of the two attempts usually wins, bounding the tail near ~2x p50.
- **Prints the recovery** - p99 with hedging drops from 4000 ms to 900 ms, back near the budget - paired with a timeout at the budget and a fast fallback.

**In one line:** the tail is a real slice, so set a budget, measure p99, and defend it with a timeout + a hedge (2nd attempt at p50) + a fallback.

**What you'll see:** TTFT p50 400 ms vs p99 4000 ms (10x); the raw p99 breaches the 800 ms SLO by 5x; and with a hedge fired at p50 the p99 falls from 4000 ms to 900 ms - back near the budget. (The following `latency-stack.py` is an illustrative markdown snippet - streaming + cached prefix + capped output + tail guard - not a runnable cell.)

Seven cells, one idea reused seven ways: `ttft` is prefill (grows with input, sets the felt wait) and `out x TPOT` is decode (grows with output, sets the rest). Streaming collapses the felt wait to TTFT for free; caching, a shorter prompt, and a faster model cut prefill; a shorter output and speculative decoding cut decode; parallelism and a semantic cache remove waiting; and a timeout-plus-hedge defends the tail. Ask two questions of any latency change - did it cut the number the user feels (TTFT), and did it hit the phase that was actually the bottleneck. The serving-layer levers hinted at here (continuous batching, KV-cache quantization, speculative decoding on your own GPU) arrive when you self-host in Lessons 13.5 and 13.6.